# Ames Housing — Exploratory Data Analysis

Exploration of the training set only (post train/test split) to inform
preprocessing decisions for the regression pipeline.

In [ ]:
import sys
sys.path.append('..')

from src.data_loading import load_raw_data, split_features_target, train_test_split_data
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = load_raw_data("../data/raw/housing.csv")
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)
print("Shape of X_train:", X_train.shape)
print("Shape of X_test :", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test :", y_test.shape)

X_train.head()

## The target `SalePrice`

In [ ]:

y_train.hist()
print("Skew =", y_train.skew())
%matplotlib inline

**Observation:** right-skewed (skew ≈ 1.75); most houses 100k–200k,
long tail to ~700k → consider log-transforming `SalePrice` in Phase 3.

## Missing values

In [ ]:
missing_pct = X_train.isnull().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]
missing_pct

**Observations:**

- `Pool QC`, `Misc Feature`, `Alley`, `Fence`, `Mas Vnr Type`, `Fireplace Qu`
  (48–99% missing) — structural (feature absent) → "None" category
- `Lot Frontage` (16.8% missing) — ambiguous, genuine non-collection
  → median imputation
- `Garage Yr Blt`, `Garage Cond`, `Garage Finish`, `Garage Qual`, `Garage Type`
  (≈5.2% missing) — no garage → "None" / 0
- `Bsmt Exposure`, `BsmtFin Type 1/2`, `Bsmt Qual`, `Bsmt Cond`,
  `Mas Vnr Area`, `Bsmt Full/Half Bath`, `Total Bsmt SF`, `Garage Area`,
  `Garage Cars`, `Bsmt Unf SF`, `BsmtFin SF 1/2` (≤0.81% missing) —
  no basement/veneer → 0 / "None"

→ Only `Lot Frontage` needs standard imputation; the rest is structural
missingness to encode explicitly.

## Numeric feature distributions

In [ ]:
numeric_cols = X_train.select_dtypes(include='number').columns
X_train[numeric_cols].hist(figsize=(16, 12), bins=30)
plt.tight_layout()


**Observations:**

- Strong outliers: `Lot Area`, `Mas Vnr Area`, `BsmtFin SF 1/2`,
  `Total Bsmt SF`, `Garage Area` → watch for capping
- Mostly zero, low variance: `Low Qual Fin SF`, `Bsmt Half Bath`,
  `3Ssn Porch`, `Screen Porch`, `Pool Area`, `Misc Val`, `Enclosed Porch`
  → candidates to exclude/group
- `Garage Yr Blt` — abnormal axis (~2200), checked below
- `Year Built`/`Garage Yr Blt` bimodal (old vs. recent construction
  waves) — expected, not a quality issue

In [ ]:
print("Typo in data:", (X_train['Garage Yr Blt'] > 2100).any())
print("Typo value:", X_train.loc[X_train['Garage Yr Blt'] > 2100, 'Garage Yr Blt'])

**Observation:** `Garage Yr Blt` has an outlier at index 2260 (2207.0),
likely a digit inversion (2007 → 2207). → Fixed in `preprocessing.py`
via `fix_known_data_errors`, run before
`add_structural_indicators`/`fill_structural_na`.

## Categorical feature distributions

In [ ]:
categorical_cols = X_train.select_dtypes(include='object').columns

for col in categorical_cols:
    top_freq = X_train[col].value_counts(normalize=True).iloc[0]
    n_categories = X_train[col].nunique()
    if top_freq > 0.9 or n_categories > 10:
        print(f"--- {col} (top category: {top_freq:.0%}, {n_categories} categories) ---")
        print(X_train[col].value_counts())
        print()

**Observations (categorical columns, dominant category >90% or >10 categories):**

- Near-constant (>95%): `Street`, `Utilities`, `Condition 2`, `Roof Matl`,
  `Heating`, `Garage Cond`, `Land Slope` → candidates to exclude
- Moderate dominance (90–94%): `Central Air`, `Functional`, `Garage Qual`,
  `Electrical`, `Bsmt Cond`, `Paved Drive` → watch, don't exclude
- High cardinality, healthy split: `Neighborhood` (28),
  `Exterior 1st`/`Exterior 2nd` (16 each, near-redundant) → mind
  one-hot size
- `Garage Cond`/`Garage Qual` overlap with the no-garage structural
  missingness already noted → handle together

## Feature vs target relationships

In [ ]:
numeric_cols = X_train.select_dtypes(include='number').columns
n_cols = 6
n_rows = -(-len(numeric_cols) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].scatter(X_train[col], y_train, alpha=0.3, s=10)
    axes[i].set_title(col, fontsize=9)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

**Observations:**

- Strong relationships with `SalePrice`: `Overall Qual`, `Gr Liv Area`,
  `Total Bsmt SF`, `1st Flr SF`, `Garage Cars`, `Garage Area`,
  `Year Built`, `Garage Yr Blt`
- Present but scattered: `Mas Vnr Area`, `BsmtFin SF 1`, `2nd Flr SF`,
  `TotRms AbvGrd`, `Full Bath`, `Fireplaces`
- Little/no relationship: `Mo Sold`, `Yr Sold`, `Misc Val`, `Pool Area`,
  `3Ssn Porch`, `BsmtFin SF 2`, `Low Qual Fin SF`, `Overall Cond`
  (contrasts with `Overall Qual`)
- `Garage Yr Blt`: the ~2260 outlier is still visible, isolated from
  the main cloud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

X_train.assign(SalePrice=y_train).boxplot(column='SalePrice', by='Overall Qual', ax=axes[0])
axes[0].set_title('SalePrice by Overall Qual')

X_train.assign(SalePrice=y_train).boxplot(column='SalePrice', by='Neighborhood', ax=axes[1], rot=90)
axes[1].set_title('SalePrice by Neighborhood')

plt.suptitle('')
plt.tight_layout()

**Observations:**

- `Overall Qual` — near-perfect monotonic relationship with `SalePrice`
  → likely the most predictive feature
- `Neighborhood` — grouped effect: premium (`NoRidge`, `NridgHt`,
  `StoneBr`) well above, low-tier (`BrDale`, `MeadowV`, `IDOTRR`,
  `OldTown`) well below → key categorical, encode carefully (one-hot or
  group by price tier in Phase 3)

## Correlations between numeric features

In [ ]:
corr_matrix = X_train[numeric_cols].corr()

plt.figure(figsize=(16, 14))
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.tight_layout()
print(X_train['MS SubClass'].unique())

**Observations:**

- Strongly correlated pairs (>0.8): `Garage Cars`/`Garage Area`,
  `Total Bsmt SF`/`1st Flr SF`, `Year Built`/`Garage Yr Blt`,
  `Gr Liv Area`/`TotRms AbvGrd` → keep in mind for Phase 3, not
  necessarily to drop
- `Overall Qual` moderately correlated with size/age features — expected
- `MS SubClass` negatively correlated with several columns — likely
  categorical (dwelling type) encoded numerically → reclassify in
  Phase 3

## Summary — Preprocessing decisions

### Target (`SalePrice`)
- Skew ≈ 1.75 → consider log transform for linear models

### Fix before NaN/encoding handling
- `Garage Yr Blt`, index 2260: 2207.0 → 2007.0 (typo) →
  `fix_known_data_errors`, before `add_structural_indicators`/
  `fill_structural_na`
- `Unnamed: 0` (CSV index artifact) → already fixed at load
  (`index_col=0`)

### Structural missing (→ "None"/0 + presence indicator)
- `Pool QC`, `Misc Feature`, `Alley`, `Fence`, `Mas Vnr Type`,
  `Fireplace Qu` (48–99%)
- `Garage Yr Blt`, `Garage Cond`, `Garage Finish`, `Garage Qual`,
  `Garage Type` (≈5.2%)
- `Bsmt Exposure`, `BsmtFin Type 1/2`, `Bsmt Qual`, `Bsmt Cond`,
  `Mas Vnr Area`, `Bsmt Full/Half Bath`, `Total Bsmt SF`, `Garage Area`,
  `Garage Cars`, `Bsmt Unf SF`, `BsmtFin SF 1/2` (≤2.7%)

### Ambiguous missing (→ standard imputation)
- `Lot Frontage` (16.8%) → median

### Numeric columns to watch (outliers)
- `Lot Area`, `Mas Vnr Area`, `BsmtFin SF 1/2`, `Total Bsmt SF`,
  `Garage Area`

### Near-constant categoricals (exclude candidates)
- `Street`, `Utilities`, `Condition 2`, `Roof Matl`, `Heating`,
  `Garage Cond`, `Land Slope`

### Moderate-dominance categoricals (watch, don't exclude)
- `Central Air`, `Functional`, `Garage Qual`, `Electrical`, `Bsmt Cond`,
  `Paved Drive`

### Numeric → categorical
- `MS SubClass` — nominal dwelling-type codes → encode as categorical

### Strongly correlated pairs (keep in mind, don't drop by default)
- `Garage Cars`/`Garage Area`, `Total Bsmt SF`/`1st Flr SF`,
  `Year Built`/`Garage Yr Blt`, `Gr Liv Area`/`TotRms AbvGrd`,
  `Exterior 1st`/`Exterior 2nd`

### Most promising predictive features
- `Overall Qual`, `Gr Liv Area`, `Total Bsmt SF`, `1st Flr SF`,
  `Garage Cars`, `Garage Area`, `Year Built`, `Neighborhood`